### Estudo de Caso 1: O Problema da Dieta (Pequeno Porte - 4 Variáveis)
* **Referência:** Goldbarg & Luna (2005), Capítulo 2, Seção 2.2.2.
* **Objetivo:** Determinar a quantidade de certos alimentos que devem ser ingeridos diariamente para que os requisitos nutricionais mínimos sejam satisfeitos ao **custo mínimo**[cite: 30].

#### Tabela de Nutrientes na Dieta Alimentar
| Vitamina | Leite (litro) | Carne (kg) | Peixe (kg) | Salada (100g) | Requisito Nutricional Mínimo |
| :---: | :---: | :---: | :---: | :---: | :---: |
| **A** | 2 mg| 2 mg| 10 mg| 20 mg| **11 mg**|
| **C** | 50 mg| 20 mg| 10 mg| 30 mg| **70 mg** |
| **D** | 80 mg| 70 mg| 10 mg| 80 mg| **250 mg**|
| **Custo** | R$ 2,00| R$ 4,00| R$ 1,50| R$ 1,00| — |

* **Regras do Problema:**
  - As restrições garantem que as somas das vitaminas fornecidas por cada alimento atendam ou superem as necessidades diárias mínimas.


In [1]:
##Import
import gurobipy as gp
from gurobipy import GRB

In [2]:
#definindo o modelo
try:
    m = gp.Model("Problema_da_Dieta") 
except gp.GurobiError as e:
    print(f"Erro no Gurobi: {e}")    

Restricted license - for non-production use only - expires 2027-11-29


In [3]:
## Definindo variaveis de descisão:
x_leite = m.addVar(lb=0.0, vtype=GRB.CONTINUOUS, name="Leite_litros")
x_carne = m.addVar(lb=0.0, vtype=GRB.CONTINUOUS, name="Carne_kg")
x_peixe = m.addVar(lb=0.0, vtype=GRB.CONTINUOUS, name="Peixe_kg")
x_salada = m.addVar(lb=0.0, vtype=GRB.CONTINUOUS, name="Salada_100g")
print(m)
print(x_leite)

<gurobi.Model Continuous instance Problema_da_Dieta: 0 constrs, 0 vars, No parameter changes>
<gurobi.Var *Awaiting Model Update*>


In [4]:
## Definindo a função objetivo : z = Minimizar {f(x) = 2xl + 4xc + 1,5xp + xs}
m.setObjective(2.0 * x_leite + 4.0 * x_carne + 1.5 * x_peixe + 1.0 * x_salada, GRB.MINIMIZE)

In [5]:
## Restrições: 

# 2xl + 2xc + 10xp + 20xs >= 11
m.addConstr(2*x_leite + 2*x_carne + 10*x_peixe + 20*x_salada >= 11, "Vitamina_A")
# 50xl + 20xc + 10xp + 30xs >= 70
m.addConstr(50*x_leite + 20*x_carne + 10*x_peixe + 30*x_salada >= 70, "Vitamina_C")
# 80xl + 70xc + 10xp + 80xs >= 250
m.addConstr(80*x_leite + 70*x_carne + 10*x_peixe + 80*x_salada >= 250, "Vitamina_D")

# Não negatividade: lb=0.0



<gurobi.Constr *Awaiting Model Update*>

In [6]:
#otimizar
m.optimize()

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (linux64 - "Manjaro Linux")

CPU model: 13th Gen Intel(R) Core(TM) i7-13650HX, instruction set [SSE2|AVX|AVX2]
Thread count: 20 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 3 rows, 4 columns and 12 nonzeros (Min)
Model fingerprint: 0xb927785f
Model has 4 linear objective coefficients
Coefficient statistics:
  Matrix range     [2e+00, 8e+01]
  Objective range  [1e+00, 4e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+01, 2e+02]

Presolve removed 0 rows and 2 columns
Presolve time: 0.01s
Presolved: 3 rows, 2 columns, 6 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    0.0000000e+00   5.975000e+01   0.000000e+00      0s
       1    3.1250000e+00   0.000000e+00   0.000000e+00      0s

Solved in 1 iterations and 0.01 seconds (0.00 work units)
Optimal objective  3.125000000e+00


In [13]:
if m.status == GRB.OPTIMAL:
        print("\n" + "="*40)
        print("SOLUÇÃO ÓTIMA DA DIETA")
        print("="*40)
        print(f"Custo mínimo da dieta: R$ {m.ObjVal:.2f}\n")
        print("Quantidades ótimas a consumir:")
        print(f" - Leite: {x_leite.X:.2f} litros")
        print(f" - Carne: {x_carne.X:.2f} kg")
        print(f" - Peixe: {x_peixe.X:.2f} kg")
        print(f" - Salada: {x_salada.X:.2f} porções de 100g")


SOLUÇÃO ÓTIMA DA DIETA
Custo mínimo da dieta: R$ 3.12

Quantidades ótimas a consumir:
 - Leite: 0.00 litros
 - Carne: 0.00 kg
 - Peixe: 0.00 kg
 - Salada: 3.12 porções de 100g


In [11]:
# Lista todas as variáveis e restrições que estão carregadas no Gurobi 
print("Variáveis:", [v.varName for v in m.getVars()])
print("Restrições:", [c.constrName for c in m.getConstrs()])

Variáveis: ['Leite_litros', 'Carne_kg', 'Peixe_kg', 'Salada_100g']
Restrições: ['Vitamina_A', 'Vitamina_C', 'Vitamina_D']


* **Conclusão:**
  - O Gurobi otimiza estritamente o que foi modelado. Como não incluímos restrições de limite máximo por alimento (ex: x_salada <= 1) ou limites de calorias, 
    o solver percebeu que comer cerca de 300g de salada pura é a forma mais barata possível de bater todas as metas de vitaminas.
